# **Stock Market Data Cleaning: AAPL, MSFT, SPY (5 Years)**
Project Overview
Professional data cleaning pipeline for historical stock market OHLCV data.

Dataset: Daily OHLCV data for AAPL, MSFT, SPY over 5 years (~1,255 trading days)
Objectives: Download raw data → Clean → Optimize → Export clean CSV

Key Improvements Over Original:


*  Fixed MultiIndex column flattening bug
*   Complete data cleaning pipeline
*   Comprehensive markdown documentation
*   Memory optimization
*   Robust validation before export







# Section 1: Environment Setup
Install required libraries and configure environment for clean, reproducible output.

In [1]:
!pip install yfinance
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import yfinance as yf

# **Section 2: Download and Inspect Data**
Download 5 years of daily OHLCV data and inspect structure.

In [2]:
import yfinance as yf
import pandas as pd
tickers=['AAPL','MSFT','SPY']
data = yf.download(tickers, period="5y", interval="1d", auto_adjust=True, progress=False)
data.shape

(1255, 15)

In [3]:
data.columns = [f"{ticker}_{metric}" for metric, ticker in data.columns]
data=data.reset_index()
data.head()


,Date,AAPL_Close,MSFT_Close,SPY_Close,AAPL_High,MSFT_High,SPY_High,AAPL_Low,MSFT_Low,SPY_Low,AAPL_Open,MSFT_Open,SPY_Open,AAPL_Volume,MSFT_Volume,SPY_Volume
0,2021-07-02,136.426285,266.459839,405.368011,136.465268,266.795739,405.723176,134.272077,261.517406,402.377182,134.418283,261.824516,403.452030,78852600,26458000,57697700
1,2021-07-06,138.434265,266.469391,404.629669,139.535725,268.110464,405.639086,136.533502,263.244795,401.900563,136.533502,266.824474,405.424111,108181800,31565600,68710400
2,2021-07-07,140.919907,268.647827,406.059662,141.231820,269.377206,406.340068,139.058127,265.979872,403.302518,139.915898,268.139189,405.311969,104911600,23260000,63549500
3,2021-07-08,139.623459,266.239136,402.751099,140.422748,267.496336,403.508147,137.118339,263.791891,399.573331,138.005367,265.740075,400.750975,105575500,24618600,97595200
4,2021-07-09,141.446259,266.738129,407.050415,141.972618,266.843681,407.349504,139.048362,264.223728,402.554843,139.145843,264.607601,404.255878,99890800,23916700,76238600


**Section 2.1: Basic Understanding - Data Structure**

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1255 entries, 0 to 1254
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         1255 non-null   datetime64[ns]
 1   AAPL_Close   1255 non-null   float64       
 2   MSFT_Close   1255 non-null   float64       
 3   SPY_Close    1255 non-null   float64       
 4   AAPL_High    1255 non-null   float64       
 5   MSFT_High    1255 non-null   float64       
 6   SPY_High     1255 non-null   float64       
 7   AAPL_Low     1255 non-null   float64       
 8   MSFT_Low     1255 non-null   float64       
 9   SPY_Low      1255 non-null   float64       
 10  AAPL_Open    1255 non-null   float64       
 11  MSFT_Open    1255 non-null   float64       
 12  SPY_Open     1255 non-null   float64       
 13  AAPL_Volume  1255 non-null   int64         
 14  MSFT_Volume  1255 non-null   int64         
 15  SPY_Volume   1255 non-null   int64         
dtypes: dat

In [5]:
data.describe()

,Date,AAPL_Close,MSFT_Close,SPY_Close,AAPL_High,MSFT_High,SPY_High,AAPL_Low,MSFT_Low,SPY_Low,AAPL_Open,MSFT_Open,SPY_Open,AAPL_Volume,MSFT_Volume,SPY_Volume
count,1255,1255.000000,1255.000000,1255.000000,1255.000000,1255.000000,1255.000000,1255.000000,1255.000000,1255.000000,1255.000000,1255.000000,1255.000000,1.255000e+03,1.255000e+03,1.255000e+03
mean,2023-12-31 04:35:22.709163264,193.936989,358.830110,499.954837,195.876883,362.191699,502.616804,191.834415,355.226229,496.845073,193.743301,358.839250,499.844938,6.529992e+07,2.655887e+07,7.567682e+07
min,2021-07-02 00:00:00,122.933556,207.733978,339.378510,125.637661,213.706637,342.481430,122.097738,206.938942,331.335626,123.907026,210.933650,332.382626,1.791060e+07,5.855900e+06,2.604870e+07
25%,2022-09-29 12:00:00,156.586388,282.945831,407.022324,159.108313,284.809960,409.383994,154.520840,279.556402,403.321101,156.820398,282.560321,407.032215,4.579505e+07,1.907310e+07,5.654270e+07
50%,2023-12-29 00:00:00,182.708572,367.097504,459.779968,184.404807,369.177050,461.012733,180.846033,363.124633,456.712483,182.343679,366.416370,458.333593,5.726670e+07,2.365290e+07,7.118120e+07
75%,2025-04-01 12:00:00,226.567451,418.255997,586.652069,228.692270,421.486167,589.248592,224.440934,414.078096,583.623602,226.804801,418.199500,587.005569,7.738550e+07,3.066290e+07,8.940515e+07
max,2026-07-02 00:00:00,315.200012,538.658569,757.618225,317.399994,551.048474,758.446109,309.649994,537.366763,754.805464,314.179993,550.830186,756.201867,3.186799e+08,1.862016e+08,2.566114e+08
std,NaN,44.921082,83.622142,112.043144,45.307600,83.966630,112.174478,44.554200,83.293173,111.795483,44.917991,83.746609,112.054373,2.873440e+07,1.205648e+07,2.911343e+07


# **Section 3: Data Cleaning Pipeline**
Execute comprehensive cleaning process.


In [6]:
data['Date'] = pd.to_datetime(data['Date'])

In [7]:
dup=data.duplicated().sum()
dup

np.int64(0)

In [8]:
price_cols = [col for col in data.columns if any(metric in col for metric in ['Open', 'High', 'Low', 'Close', 'Volume'])]
data[price_cols] = data[price_cols].ffill()
data[price_cols] = data[price_cols].bfill()

Initial inspection (data.isnull().sum()) revealed zero missing values in the raw dataset. However, .ffill() and .bfill() have been retained in the pipeline as a defensive programming measure to ensure the pipeline remains robust against unexpected data gaps in future data downloads.

In [9]:
data.isnull().sum()

,0
Date,0
AAPL_Close,0
MSFT_Close,0
SPY_Close,0
AAPL_High,0
MSFT_High,0
SPY_High,0
AAPL_Low,0
MSFT_Low,0
SPY_Low,0


In [10]:
data

,Date,AAPL_Close,MSFT_Close,SPY_Close,AAPL_High,MSFT_High,SPY_High,AAPL_Low,MSFT_Low,SPY_Low,AAPL_Open,MSFT_Open,SPY_Open,AAPL_Volume,MSFT_Volume,SPY_Volume
0,2021-07-02,136.426285,266.459839,405.368011,136.465268,266.795739,405.723176,134.272077,261.517406,402.377182,134.418283,261.824516,403.452030,78852600,26458000,57697700
1,2021-07-06,138.434265,266.469391,404.629669,139.535725,268.110464,405.639086,136.533502,263.244795,401.900563,136.533502,266.824474,405.424111,108181800,31565600,68710400
2,2021-07-07,140.919907,268.647827,406.059662,141.231820,269.377206,406.340068,139.058127,265.979872,403.302518,139.915898,268.139189,405.311969,104911600,23260000,63549500
3,2021-07-08,139.623459,266.239136,402.751099,140.422748,267.496336,403.508147,137.118339,263.791891,399.573331,138.005367,265.740075,400.750975,105575500,24618600,97595200
4,2021-07-09,141.446259,266.738129,407.050415,141.972618,266.843681,407.349504,139.048362,264.223728,402.554843,139.145843,264.607601,404.255878,99890800,23916700,76238600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1250,2026-06-26,283.779999,372.970001,728.989990,285.950012,376.609985,736.530029,274.209991,355.429993,716.580017,275.000000,357.149994,728.950012,261775500,186201600,71034000
1251,2026-06-29,281.739990,368.570007,741.000000,288.369995,380.500000,741.559998,279.850006,359.899994,732.090027,286.730011,377.500000,736.530029,66427000,51229900,58035200
1252,2026-06-30,289.359985,373.019989,746.770020,289.940002,374.149994,748.020020,280.700012,367.450012,740.890015,281.170013,371.029999,741.289978,65100200,44945700,55626000
1253,2026-07-01,294.380005,384.279999,745.760010,296.589996,388.829987,749.440002,289.200012,374.890015,742.380005,293.440002,380.829987,745.000000,50109500,47966000,46886600


# **Pre-column cleaning notes**



In [11]:
cleaning_notes = {
    'Date': {
        'Type': 'datetime64[ns]',
        'Issue Found': 'None',
        'Action Taken': 'Parsed with pd.to_datetime()'
    },
    'AAPL_Close / MSFT_Close / SPY_Close': {
        'Type': 'float32',
        'Issue Found': '0 nulls',
        'Action Taken': 'fill/bfill (no-op), cast to float32'
    },
    'AAPL_High / MSFT_High / SPY_High': {
        'Type': 'float32',
        'Issue Found': '0 nulls',
        'Action Taken': 'Price integrity validated, cast to float32'
    },
    'AAPL_Low / MSFT_Low / SPY_Low': {
        'Type': 'float32',
        'Issue Found': '0 nulls',
        'Action Taken': 'Price integrity validated, cast to float32'
    },
    'AAPL_Open / MSFT_Open / SPY_Open': {
        'Type': 'float32',
        'Issue Found': '0 nulls',
        'Action Taken': 'Price integrity validated, cast to float32'
    },
    'AAPL_Volume / MSFT_Volume / SPY_Volume': {
        'Type': 'int32',
        'Issue Found': '0 nulls',
        'Action Taken': 'Cast to int32 (logically whole numbers, yfinance returns float64)'
    }
}

notes_df = pd.DataFrame(cleaning_notes).T
notes_df

,Type,Issue Found,Action Taken
Date,datetime64[ns],None,Parsed with pd.to_datetime()
AAPL_Close / MSFT_Close / SPY_Close,float32,0 nulls,"fill/bfill (no-op), cast to float32"
AAPL_High / MSFT_High / SPY_High,float32,0 nulls,"Price integrity validated, cast to float32"
AAPL_Low / MSFT_Low / SPY_Low,float32,0 nulls,"Price integrity validated, cast to float32"
AAPL_Open / MSFT_Open / SPY_Open,float32,0 nulls,"Price integrity validated, cast to float32"
AAPL_Volume / MSFT_Volume / SPY_Volume,int32,0 nulls,"Cast to int32 (logically whole numbers, yfinan..."


# **Section 4: Price Integrity Validation**
What we're checking:
For any given trading day, the relationship between OHLC prices should satisfy: Low ≤ Open, Close ≤ High

Why this matters:
Violations indicate data corruption, entry errors, or outliers that could distort analysis.

Approach:
For each stock, we validate that daily price relationships are logically consistent.

In [12]:
tickers_in_data = list(set([col.split('_')[0] for col in price_cols]))
price_issues = {}
for ticker in sorted(tickers_in_data):
    open_col = f"{ticker}_Open"
    high_col = f"{ticker}_High"
    low_col = f"{ticker}_Low"
    close_col = f"{ticker}_Close"

    if all(col in data.columns for col in [open_col, high_col, low_col, close_col]):
        issue1 = ((data[low_col] > data[open_col]) | (data[open_col] > data[high_col])).sum()
        issue2 = ((data[low_col] > data[close_col]) | (data[close_col] > data[high_col])).sum()
        issue3 = (data[low_col] > data[high_col]).sum()
        total_issues = issue1 + issue2 + issue3
        price_issues[ticker] = total_issues
        if total_issues == 0:
            print(f"All {len(data)} records satisfy Low ≤ Open,Close ≤ High")
        else:
            print(f"Found {total_issues} price relationship violations")
            if issue1 > 0:
                print(f"Low > Open OR Open > High: {issue1} records")
            if issue2 > 0:
                print(f"Low > Close OR Close > High: {issue2} records")
            if issue3 > 0:
                print(f"Low > High: {issue3} records")

All 1255 records satisfy Low ≤ Open,Close ≤ High
All 1255 records satisfy Low ≤ Open,Close ≤ High
All 1255 records satisfy Low ≤ Open,Close ≤ High


# **Section 5: Memory Optimization**


In [13]:
memory_before = data.memory_usage(deep=True).sum() / 1024**2


In [14]:
float_cols = data.select_dtypes(include=['float64']).columns
for col in float_cols:
    data[col] = data[col].astype('float32')

In [15]:
int_cols = data.select_dtypes(include=['int64']).columns
for col in int_cols:
    if (data[col].min() >= -2**31) and (data[col].max() <= 2**31 - 1):
        data[col] = data[col].astype('int32')

In [16]:
memory_after = data.memory_usage(deep=True).sum() / 1024**2
memory_saved = memory_before - memory_after
memory_saved_pct = (memory_saved / memory_before) * 100

# **Section 6: Validation and Export**
Validate cleaned data and export to CSV.

In [18]:
def validate(df, filename="clean_stock_data.csv"):
  print("starting to validate the data.")
  missing_vals=df.isnull().sum().sum()
  if missing_vals>0:
    print(f"Validation failed: found {missing_vals} missing values")
    return False
  duplicate_rows=df.duplicated().sum()
  if duplicate_rows>0:
    print(f"Validation failed: found {duplicate_rows} duplicate rows")
    return False
  price_cols=[col for col in df.columns if 'Close' in col or 'Open' in col or 'High' in col or 'Volume' in col or 'Low' in col]
  for col in price_cols:
    if(df[col]<=0).any():
      print(f"Validation failed: column {col} contains negative or zero prices.")
      return False # Added return False here
  if 'index' in df.columns:
        print(" Validation Failed: The unwanted 'index' column is still present.")
        return False
  print("All validations passed! Data is clean.")
  df.to_csv(filename, index=False)
  print(f"File successfully saved as '{filename}'")
  return True
validate(data)

starting to validate the data.
All validations passed! Data is clean.
File successfully saved as 'clean_stock_data.csv'


True

In [20]:
import os
output_filename = "clean_stock_data.csv"
data.to_csv(output_filename, index=False)

file_size = os.path.getsize(output_filename) / 1024